In [ ]:
import torch
import dnnlib
import legacy
import math
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import os
from sklearn.decomposition import PCA

In [ ]:
def plot_tensor_images(images, titles=None, cols=4, figsize=4):
    if not isinstance(images, list):
        images = [images]

    n = len(images)
    rows = math.ceil(n / cols)

    plt.figure(figsize=(cols * figsize, rows * figsize))

    for i, img in enumerate(images):
        img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
        pil_img = Image.fromarray(img[0].cpu().numpy(), "RGB")
        plt.subplot(rows, cols, i + 1)
        plt.imshow(pil_img)
        plt.axis("off")

        if titles and i < len(titles):
            plt.title(titles[i], fontsize=10)

    plt.tight_layout()
    plt.show()

In [ ]:
network_pkl = "ffhq-256.pkl"
device = torch.device("cuda")
with dnnlib.util.open_url(network_pkl) as f:
    G = legacy.load_network_pkl(f)["G_ema"].to(device)

In [ ]:
print(G)

In [ ]:
affines = []

for name, module in G.synthesis.named_modules():
    if hasattr(module, "affine"):
        affines.append(module.affine)
print(len(affines))
print(affines)

In [ ]:
print(affines[0].weight.shape)

In [ ]:
mapping = G.mapping

In [ ]:
z = torch.randn(1, 512).to(device)
ws = G.mapping(z, None)
print(ws.shape)

In [ ]:
class StyleAffineMapper(torch.nn.Module):
    def __init__(self, mapping, affines):
        super().__init__()
        self.mapping = mapping
        self.affines = torch.nn.ModuleList(affines)

    def forward(self, z, truncation=0.5):
        w = self.mapping(z, None, truncation_psi=truncation)  # shape [batch, 14, 512]

        outputs = []

        # Handle First Block
        outputs.append(self.affines[0](w[:, 0]))  # conv1
        outputs.append(self.affines[1](w[:, 1]))  # toRGB

        # Rest of the blocks
        w_idx = 2
        affine_idx = 2
        for _ in range(6):
            # conv0
            outputs.append(self.affines[affine_idx](w[:, w_idx]))
            # conv1
            outputs.append(self.affines[affine_idx + 1](w[:, w_idx + 1]))
            # toRGB (reuses the second w vector of this block)
            outputs.append(self.affines[affine_idx + 2](w[:, w_idx + 1]))

            w_idx += 2
            affine_idx += 3

        return outputs

In [ ]:
class StyleSynthesisNetwork(torch.nn.Module):
    def __init__(self, synthesis):
        super().__init__()
        self.synthesis = synthesis
        
        for name, module in self.synthesis.named_modules():
            if hasattr(module, 'affine'):
                module.affine = torch.nn.Identity()

    def forward(self, precomputed_styles):

        style_idx = 0
        
        def hooked_forward(module, input):
            nonlocal style_idx
            new_input = list(input)
            new_input[1] = precomputed_styles[style_idx]
            style_idx += 1
            return tuple(new_input)

        hooks = []
        for name, module in self.synthesis.named_modules():
            if hasattr(module, 'affine'):
                hooks.append(module.register_forward_pre_hook(hooked_forward)) # pre-hook to replace the style input with precomputed styles

        try:
            dummy_ws = torch.zeros(precomputed_styles[0].shape[0],
                                   self.synthesis.num_ws, 512).to(precomputed_styles[0].device) # use a dummy tensor since styles are loaded via hooks
            img = self.synthesis(dummy_ws)
        finally:
            for h in hooks:
                h.remove()
                
        return img

In [ ]:
style_generator = StyleAffineMapper(mapping, affines)
style_synthesis = StyleSynthesisNetwork(G.synthesis)

z = torch.randn(1, 512).to(device)
styles = style_generator(z, truncation=0.5)
for i, style_vector in enumerate(styles):
    print(f"Style vector {i+1}: shape {style_vector.shape}")
generated_image = style_synthesis(styles)
print(generated_image.shape)
plot_tensor_images(generated_image, titles=["Generated Image"], cols=1, figsize=6)

In [ ]:
zs = torch.randn(50000, 512).to(device)
style_outputs = style_generator(zs, truncation=0.5)
flat_tensor = torch.cat(style_outputs, dim=1).detach().cpu().numpy()
print(flat_tensor.shape)

In [ ]:
pca = PCA(n_components=100)  # first 100 principal components

print(flat_tensor.shape) # confirm shape
X_pca = pca.fit_transform(flat_tensor)

principal_axes = pca.components_

print(principal_axes[0].shape)

In [ ]:
def generate(styles,filename="sample",folder="out"):
    try:
        
        # Generate Image
        img = style_synthesis(styles)

        # Process Image (Tensor to uint8)
        img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
        pil_img = Image.fromarray(img[0].cpu().numpy())

        # Save to Disk
        filepath = f"{folder}/{filename}.png"
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        pil_img.save(filepath)

        print("Saved to: ", filepath)

    except Exception as e:
        raise Exception(e)


In [ ]:
def flattened_to_style_vectors(flattened_vector):
    style = []
    total = 0
    for i in range(20):
        if i < 12:
            style.append(torch.tensor(flattened_vector[total:total+512], dtype=torch.float32).unsqueeze(0).to(device))
            total += 512
        elif i >= 12 and i < 15:
            style.append(torch.tensor(flattened_vector[total:total+256], dtype=torch.float32).unsqueeze(0).to(device))
            total += 256
        elif i >= 15 and i < 18:
            style.append(torch.tensor(flattened_vector[total:total+128], dtype=torch.float32).unsqueeze(0).to(device))
            total += 128
        else:
            style.append(torch.tensor(flattened_vector[total:total+64], dtype=torch.float32).unsqueeze(0).to(device))
            total += 64
    return style
   
if torch.cuda.is_available():
    torch.cuda.empty_cache()

z = torch.randn(1, 512).to(device)
style_avg = style_generator(z, truncation=0.0)

for j in range(20):
  pc1 = principal_axes[j]
  principle_style = flattened_to_style_vectors(pc1)

  for i in range(10):
      alpha = (i-5) * 20
      style_step = [p * alpha for p in principle_style]

      style_tensor = [a + b for a, b in zip(style_avg, style_step)]
      generate(style_tensor ,filename=i ,folder=f"out/{j}")